In [1]:
import numpy as np
import math
from scipy.optimize import brentq


def _normalize_p(p, K=None):
    """Normalize p to a probability vector of length K."""
    p = np.asarray(p, dtype=float)
    
    if K is not None and len(p) != K:
        raise ValueError(f"len(p)={len(p)} != K={K}")
    s = p.sum()
    if s <= 0:
        raise ValueError("p must sum to a positive number")
    return p / s


def _choose_p(p_desc, K, rng=None):
    """
    p_desc:
      - 'uniform'
      - float: Dirichlet alpha
      - array-like: explicit p
    """
    if rng is None:
        rng = np.random.default_rng(0)

    if p_desc == "uniform":
        return np.full(K, 1.0 / K, dtype=float)
    if isinstance(p_desc, (list, tuple, np.ndarray)):
        return _normalize_p(p_desc, K=K)
    if isinstance(p_desc, (float, int)):
        alpha = float(p_desc)
        if alpha <= 0:
            raise ValueError("Dirichlet alpha must be > 0")
        return rng.dirichlet([alpha] * K)
    raise ValueError("Unsupported p_desc. Use 'uniform', a float alpha, or an explicit p vector.")


def _binom_pmf_all(C, p, m_max):
    """
    Return pmf[0..m_max] and tail_prob = P(m > m_max) for Binomial(C,p).
    Use stable recurrence to avoid large combinations.
    """
    C = int(C)
    if C < 0:
        raise ValueError("C must be >= 0")
    if not (0 <= p <= 1):
        raise ValueError("p must be in [0,1]")
    if C == 0:
        pmf = np.zeros(m_max + 1, dtype=float)
        pmf[0] = 1.0
        return pmf, 0.0

    m_max_eff = min(m_max, C)
    pmf = np.zeros(m_max + 1, dtype=float)

    # P(0) = (1-p)^C
    pm0 = (1 - p) ** C
    pmf[0] = pm0

    # Recurrence: P(m+1) = P(m) * (C-m)/(m+1) * p/(1-p)
    if p == 0:
        tail = 0.0
        return pmf, tail
    if p == 1:
        # all mass at C
        if C <= m_max:
            pmf[C] = 1.0
            tail = 0.0
        else:
            tail = 1.0
        return pmf, tail

    ratio = p / (1 - p)
    pm = pm0
    for m in range(0, m_max_eff):
        pm = pm * (C - m) / (m + 1) * ratio
        pmf[m + 1] = pm

    # Tail prob beyond m_max
    tail = 1.0 - pmf[: m_max_eff + 1].sum()
    # If m_max > C, tail should be 0
    if m_max >= C:
        tail = 0.0
    # numerical guard
    tail = max(0.0, min(1.0, tail))
    return pmf, tail


def metrics_multinomial(C, D, K, p_desc="uniform", m_max=40, rng=None):
    """
    Multinomial-only model:
      - Cells -> droplets: (N1..ND) ~ Multinomial(C; 1/D,...,1/D)
        => Nj marginal ~ Binomial(C, 1/D)
      - Within droplet: (n_{j1}..n_{jK}) | Nj=m ~ Multinomial(m; p)

    Returns rates with denominator = expected nonempty droplets.
      nonempty_rate: E[#nonempty]/D
      doublet_rate:  E[#multiplet]/E[#nonempty]
      SSM_rate:      E[#(multiplet & same-sample)]/E[#nonempty]
      MSM_rate:      E[#(multiplet & mixed-sample)]/E[#nonempty]
    """
    C = int(round(C))
    D = int(round(D))
    if C < 0 or D <= 0 or K <= 0:
        raise ValueError("Require C>=0, D>0, K>0")

    p = _choose_p(p_desc, K, rng=rng)

    # Nj ~ Binomial(C, 1/D)
    prob = 1.0 / D
    pmf, tail = _binom_pmf_all(C, prob, m_max=m_max)

    # E[#nonempty] = D * P(Nj>=1)
    p0 = pmf[0]
    p_nonempty = 1.0 - p0
    if p_nonempty <= 0:
        # no nonempty droplets
        return {
            "C": C, "D": D, "K": K, "p_desc": p_desc,
            "nonempty_rate": 0.0,
            "doublet_rate": 0.0,
            "SSM_rate": 0.0,
            "MSM_rate": 0.0,
        }

    # P(Nj=1)
    p1 = pmf[1] if C >= 1 else 0.0

    # P(Nj>=2)
    p_ge2 = 1.0 - p0 - p1

    # Expected-rate version under nonempty denominator:
    doublet_rate = p_ge2 / p_nonempty

    # Need P(multiplet & same)/P(nonempty)
    # For each m>=2: P(Nj=m) * P(same|m), where P(same|m)=sum_k p_k^m
    # Use truncated sum up to m_max, and approximate tail conservatively.
    same_sum = 0.0
    mixed_sum = 0.0

    # Precompute sum p_k^m iteratively for speed
    # powers[k] = p_k^m
    pk = p.copy()
    powers = pk.copy()  # m=1
    # For m=2..m_max:
    for m in range(2, min(m_max, C) + 1):
        powers *= pk  # now p^m
        same_prob_m = float(powers.sum())
        pm = pmf[m]
        same_sum += pm * same_prob_m
        mixed_sum += pm * (1.0 - same_prob_m)

    # Tail handling: m > m_max
    # For SSM/MSM rates we need tail weighted by same_prob_m which decreases with m (if K>1).
    # Use a safe approximation:
    #   - same_prob is at most max(p)^m; for large m it's tiny unless max(p)~1
    # Here we ignore same in the tail (upper-bounds MSM a bit) unless K==1.
    if tail > 0:
        if K == 1:
            # all same
            same_sum += tail
        else:
            mixed_sum += tail  # assume mixed dominates; good enough for design

    SSM_rate = same_sum / p_nonempty
    MSM_rate = mixed_sum / p_nonempty

    # Numerical guard: enforce SSM+MSM≈doublet (within tolerance)
    # (May differ slightly due to tail approximation; that's okay for design.)
    return {
        "C": C, "D": D, "K": K, "p_desc": p_desc,
        "nonempty_rate": p_nonempty,
        "doublet_rate": doublet_rate,
        "SSM_rate": SSM_rate,
        "MSM_rate": MSM_rate,
    }


In [2]:
def solve_C_for_target_doublet(target_doublet, D, K, p_desc="uniform",
                               C_min=0, C_max=2_000_000, m_max=60,
                               bracket_expand=1.6, verbose=False):
    """
    Solve C (integer) for a given D such that doublet_rate ~= target_doublet
    under multinomial-only model.
    """
    if not (0.0 < target_doublet < 1.0):
        raise ValueError("target_doublet must be in (0,1)")

    D = int(round(D))

    def f(C):
        met = metrics_multinomial(C, D, K, p_desc=p_desc, m_max=m_max)
        return met["doublet_rate"] - target_doublet

    # Start with bounds
    lo = int(max(0, C_min))
    hi = int(max(lo + 1, min(C_max, max(10, D))))  # initial guess

    flo = f(lo)
    fhi = f(hi)

    # Expand upper bound until sign changes or reach C_max
    while flo * fhi > 0 and hi < C_max:
        hi_new = int(min(C_max, max(hi + 1, int(hi * bracket_expand))))
        if hi_new == hi:
            break
        hi = hi_new
        fhi = f(hi)
        if verbose:
            print(f"[bracket] lo={lo} flo={flo:.4g} hi={hi} fhi={fhi:.4g}")

    if flo * fhi > 0:
        # No root in range: return best effort (closest)
        # Evaluate a few points to pick closest
        candidates = [lo, hi, int((lo + hi) / 2)]
        best = None
        best_abs = float("inf")
        for c in candidates:
            val = abs(f(c))
            if val < best_abs:
                best_abs = val
                best = c
        return best, False  # not guaranteed achievable

    # Solve in real domain then round to integer and pick best of neighbors
    root = brentq(lambda x: f(int(round(x))), lo, hi)  # robust but a bit hacky
    C0 = int(round(root))
    neighbors = [max(0, C0 - 1), C0, C0 + 1]
    bestC = None
    bestAbs = float("inf")
    for c in neighbors:
        val = abs(f(c))
        if val < bestAbs:
            bestAbs = val
            bestC = c

    return bestC, True


In [3]:
def generate_design_table(target_doublet=None, K=12, p_desc="uniform",
                          D_tiers=(10, 20, 50, 100),
                          C_max=2_000_000, m_max=60,
                          top_n=None):
    """
    Generate a list of dict rows. If target_doublet is provided:
      For each D in D_tiers, solve recommended C and compute metrics.
    """
    rows = []
    for D in D_tiers:
        if target_doublet is not None:
            C_sol, ok = solve_C_for_target_doublet(
                target_doublet=target_doublet,
                D=D, K=K, p_desc=p_desc,
                C_min=0, C_max=C_max, m_max=m_max
            )
            met = metrics_multinomial(C_sol, D, K, p_desc=p_desc, m_max=m_max)
            row = {
                "status": "OK" if ok else "APPROX",
                "target_doublet": target_doublet,
                **met,
                "err_doublet": met["doublet_rate"] - target_doublet,
            }
            rows.append(row)
        else:
            # If no target provided, just compute metrics at some default C if desired
            pass

    # Sort by absolute error if target is given
    if target_doublet is not None:
        rows.sort(key=lambda r: abs(r["err_doublet"]))

    if top_n is not None:
        rows = rows[:top_n]
    return rows


In [4]:
design_table = generate_design_table(
    target_doublet=0.05,
    K=12,
    p_desc="uniform",
    D_tiers=[10, 20, 50, 100],
    m_max=80
)

for r in design_table:
    print(r)


{'status': 'OK', 'target_doublet': 0.05, 'C': 6, 'D': 50, 'K': 12, 'p_desc': 'uniform', 'nonempty_rate': 0.11415761913600009, 'doublet_rate': 0.04981816862547585, 'SSM_rate': 0.0040490611768520665, 'MSM_rate': 0.0457691074486229, 'err_doublet': -0.0001818313745241551}
{'status': 'OK', 'target_doublet': 0.05, 'C': 11, 'D': 100, 'K': 12, 'p_desc': 'uniform', 'nonempty_rate': 0.10466174574128362, 'doublet_rate': 0.04949007350898785, 'SSM_rate': 0.004010581859093052, 'MSM_rate': 0.04547949164989422, 'err_doublet': -0.0005099264910121554}
{'status': 'OK', 'target_doublet': 0.05, 'C': 3, 'D': 20, 'K': 12, 'p_desc': 'uniform', 'nonempty_rate': 0.1426250000000001, 'doublet_rate': 0.050832602979843024, 'SSM_rate': 0.00416910117830363, 'MSM_rate': 0.04666350180153857, 'err_doublet': 0.000832602979843021}
{'status': 'OK', 'target_doublet': 0.05, 'C': 2, 'D': 10, 'K': 12, 'p_desc': 'uniform', 'nonempty_rate': 0.18999999999999995, 'doublet_rate': 0.05263157894736804, 'SSM_rate': 0.00438596491228070

In [5]:
def solve_C_for_target_SSM(target_SSM, D, K, p_desc="uniform",
                           C_min=0, C_max=2_000_000, m_max=80,
                           bracket_expand=1.6, verbose=False):
    """
    Solve C (integer) for a given D such that SSM_rate ~= target_SSM
    under multinomial-only model (nonempty denominator).

    Returns: (C_best, ok)
      ok=True means a sign-changing bracket was found and brentq succeeded.
      ok=False means target not bracketed; we return best effort among a few points.
    """
    if not (0.0 <= target_SSM < 1.0):
        raise ValueError("target_SSM must be in [0,1)")

    D = int(round(D))

    def f(C):
        met = metrics_multinomial(C, D, K, p_desc=p_desc, m_max=m_max)
        return met["SSM_rate"] - target_SSM

    lo = int(max(0, C_min))
    hi = int(max(lo + 1, min(C_max, max(10, D))))

    flo = f(lo)
    fhi = f(hi)

    # Expand until sign change
    while flo * fhi > 0 and hi < C_max:
        hi_new = int(min(C_max, max(hi + 1, int(hi * bracket_expand))))
        if hi_new == hi:
            break
        hi = hi_new
        fhi = f(hi)
        if verbose:
            print(f"[bracket-SSM] lo={lo} flo={flo:.4g} hi={hi} fhi={fhi:.4g}")

    if flo * fhi > 0:
        # no bracket; best effort
        candidates = [lo, hi, int((lo + hi) / 2)]
        best = None
        best_abs = float("inf")
        for c in candidates:
            val = abs(f(c))
            if val < best_abs:
                best_abs = val
                best = c
        return best, False

    # Solve in integer neighborhood
    root = brentq(lambda x: f(int(round(x))), lo, hi)
    C0 = int(round(root))
    neighbors = [max(0, C0 - 1), C0, C0 + 1]
    bestC = None
    bestAbs = float("inf")
    for c in neighbors:
        val = abs(f(c))
        if val < bestAbs:
            bestAbs = val
            bestC = c
    return bestC, True


In [6]:
def _passes_constraints(met, constraints):
    """
    constraints:
      - min_C
      - max_C
      - min_nonempty_rate
      - min_expected_nonempty  (D * nonempty_rate)
    """
    if constraints is None:
        return True
    C, D = met["C"], met["D"]
    if "min_C" in constraints and C < constraints["min_C"]:
        return False
    if "max_C" in constraints and C > constraints["max_C"]:
        return False
    if "min_nonempty_rate" in constraints and met["nonempty_rate"] < constraints["min_nonempty_rate"]:
        return False
    if "min_expected_nonempty" in constraints and D * met["nonempty_rate"] < constraints["min_expected_nonempty"]:
        return False
    return True


def generate_design_table_v2(
    K=12,
    p_desc="uniform",
    D_tiers=(10, 20, 50, 100),
    target_doublet=None,
    target_SSM=None,
    weights=(1.0, 1.0),  # (w_doublet, w_ssm)
    constraints=None,
    C_max=2_000_000,
    m_max=80,
    top_n=None,
):
    """
    Produce candidate designs with optional constraints.

    If target_doublet is given:
      - for each D: solve C for target_doublet
    If target_SSM is given:
      - for each D: solve C for target_SSM
    If both are given:
      - for each D: generate 2 candidate C's (one from each solve)
        then evaluate both and keep the better one(s).
      - rank by weighted normalized absolute error.

    Returns list[dict]
    """
    rows = []
    w_d, w_s = weights

    def add_row(D, C, ok_flag, source):
        met = metrics_multinomial(C, D, K, p_desc=p_desc, m_max=m_max)
        met.update({
            "p_desc": p_desc,
            "target_doublet": target_doublet,
            "target_SSM": target_SSM,
            "solve_source": source,
        })

        # errors
        if target_doublet is not None:
            met["err_doublet"] = met["doublet_rate"] - target_doublet
        else:
            met["err_doublet"] = None

        if target_SSM is not None:
            met["err_SSM"] = met["SSM_rate"] - target_SSM
        else:
            met["err_SSM"] = None

        # composite score for sorting (smaller is better)
        score = 0.0
        if target_doublet is not None:
            score += w_d * abs(met["err_doublet"])
        if target_SSM is not None:
            score += w_s * abs(met["err_SSM"])
        met["score"] = score

        met["status"] = "OK" if ok_flag else "APPROX"

        if _passes_constraints(met, constraints):
            rows.append(met)

    for D in D_tiers:
        D = int(round(D))
        candidates = []

        if target_doublet is not None:
            C_d, ok_d = solve_C_for_target_doublet(
                target_doublet=target_doublet, D=D, K=K, p_desc=p_desc,
                C_min=0, C_max=C_max, m_max=m_max
            )
            candidates.append((C_d, ok_d, "solve_doublet"))

        if target_SSM is not None:
            C_s, ok_s = solve_C_for_target_SSM(
                target_SSM=target_SSM, D=D, K=K, p_desc=p_desc,
                C_min=0, C_max=C_max, m_max=m_max
            )
            candidates.append((C_s, ok_s, "solve_SSM"))

        # de-duplicate C values
        seen = set()
        for C, ok, src in candidates:
            C = int(round(C))
            if C in seen:
                continue
            seen.add(C)
            add_row(D, C, ok, src)

    # sort
    rows.sort(key=lambda r: r["score"])

    if top_n is not None:
        rows = rows[:top_n]
    return rows


In [7]:
rows = generate_design_table_v2(
    K=12,
    p_desc="uniform",
    D_tiers=[100, 500, 1000, 5000],
    target_doublet=0.10,
    target_SSM=0.02,
    weights=(1.0, 2.0),  # 你更重视 SSM 的话把 w_s 调大
    constraints={"min_expected_nonempty": 500, "min_C": 1000},
    top_n=10
)
for r in rows:
    print(r["D"], r["C"], r["doublet_rate"], r["SSM_rate"], r["err_doublet"], r["err_SSM"], r["score"], r["solve_source"])


5000 1037 0.10003765698707447 0.007816115905008135 3.765698707446086e-05 -0.012183884094991866 0.024405425177058192 solve_doublet
5000 3358 0.29844836283040244 0.020001069575536927 0.19844836283040243 1.0695755369265203e-06 0.1984505019814763 solve_SSM
